# BOA-SARIMA Forecaster â€” End-to-End Demo

This notebook demonstrates the complete forecasting pipeline:

1. **Generate** synthetic monthly sales data
2. **Preprocess** â€” fill missing months, remove zero-demand series
3. **Standardise** â€” apply weighted moving-average outlier clipping
4. **Optimise** â€” find the best `ARIMA(p, d, q)` via Bayesian Optimisation (Optuna TPE)
5. **Forecast** â€” generate a 12-month outlook with confidence intervals
6. **Visualise** â€” plot historical data, forecast, and optimisation convergence

---

> **No real data required.** All data used here is generated synthetically.
> To use your own Excel file, replace the data generation cell with a call to
> `sarima_bayes.data_loader.load_data("path/to/sales.xlsx")`.

In [ ]:
# Uncomment if running in Google Colab
# !pip install git+https://github.com/TomCardeLo/boa-sarima-forecaster.git#egg=boa-forecaster[ml]

## 0. Setup

Add the `src/` directory to the Python path so the package can be imported
without a full `pip install`.  Alternatively, run `pip install -e .` from
the repository root.

In [ ]:
import os
import sys

# Add src/ to path if not already installed
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import logging
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# sarima_bayes package
from sarima_bayes import forecast_arima, optimize_arima
from sarima_bayes.model import pred_arima
from sarima_bayes.preprocessor import clean_zeros, fill_blanks
from sarima_bayes.standardization import weighted_moving_stats

print("All imports OK.")

## 1. Generate Synthetic Sales Data

We generate 36 months of synthetic monthly sales for **3 SKUs** across **2 countries**
(6 independent time series).  Each series has:

- A mild upward **trend** (~2 % per year)
- **Seasonality** peaking in Q4 (months 10â€“12)
- Gaussian **noise**

This mimics the structure of real consumer-goods demand data.

### Expected DataFrame structure

| Column    | Type       | Description                              |
|-----------|------------|------------------------------------------|
| `Date`    | datetime   | Month-start timestamp (e.g. 2022-01-01)  |
| `SKU`     | int        | Product identifier                       |
| `CS`      | float      | Sales volume (case-equivalent units)     |
| `Country` | str        | Market code (e.g. `"US"`, `"MX"`)       |

In [ ]:
def generate_sales_data(
    start: str = "2022-01-01",
    periods: int = 36,
    seed: int = 42,
) -> pd.DataFrame:
    """Generate synthetic monthly sales data with trend + seasonality + noise."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, periods=periods, freq="MS")
    months = np.arange(1, periods + 1)

    # Series definitions: (SKU, Country, base_demand, seasonal_amplitude, trend_rate)
    series_specs = [
        (1001, "US", 150.0, 35.0, 0.002),
        (1001, "MX",  80.0, 18.0, 0.001),
        (1002, "US", 200.0, 50.0, 0.003),
        (1002, "MX", 100.0, 22.0, 0.002),
        (1003, "US",  50.0, 12.0, 0.004),
        (1003, "MX",  30.0,  8.0, 0.001),
    ]

    rows = []
    for sku, country, base, amp, trend in series_specs:
        # Trend component: compound growth
        trend_comp = base * (1 + trend) ** months
        # Seasonality: sine wave peaking in October (month 10 of calendar year)
        calendar_month = pd.DatetimeIndex(dates).month
        seasonal_comp = amp * np.sin(2 * np.pi * (calendar_month - 4) / 12)
        # Gaussian noise scaled to ~5% of base
        noise = rng.normal(0, base * 0.05, size=periods)
        demand = np.maximum(trend_comp + seasonal_comp + noise, 0.0)

        for date, val in zip(dates, demand):
            rows.append({"Date": date, "SKU": sku, "CS": round(val, 1), "Country": country})

    df = pd.DataFrame(rows)
    return df


df_raw = generate_sales_data(periods=36)
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['Date'].min().date()} â†’ {df_raw['Date'].max().date()}")
print(f"SKUs: {sorted(df_raw['SKU'].unique())}")
print(f"Countries: {sorted(df_raw['Country'].unique())}")
df_raw.head(8)

### Visualise raw series

A quick look at the six time series before any processing.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
axes = axes.flatten()

combos = df_raw.groupby(["SKU", "Country"])
for ax, ((sku, country), grp) in zip(axes, combos):
    grp_sorted = grp.sort_values("Date")
    ax.plot(grp_sorted["Date"], grp_sorted["CS"], marker="o", ms=3, lw=1.5, color="steelblue")
    ax.set_title(f"SKU {sku} â€” {country}", fontsize=10)
    ax.set_ylabel("CS (units)", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Synthetic Monthly Sales â€” Raw Data", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Preprocessing

Two operations are applied:

1. **`clean_zeros`** â€” drop any series whose total demand is zero.
2. **`fill_blanks`** â€” ensure every SKU/Country has a continuous monthly
   index with no gaps (missing months are filled with zero).

In [ ]:
# Step 1: remove zero-demand series
df_clean = clean_zeros(df_raw, group_cols=["Country", "SKU"], value_col="CS")
print(f"After clean_zeros: {len(df_clean)} rows  (was {len(df_raw)})")

# Step 2: fill missing months up to forecast horizon
FORECAST_END = "2025-12-01"   # last historical month we want in the series

df_filled = fill_blanks(
    df_clean,
    date_col="Date",
    group_cols=["Country", "SKU"],
    value_col="CS",
    end_date=FORECAST_END,
    freq="MS",
)
print(f"After fill_blanks:  {len(df_filled)} rows")
df_filled.head(6)

## 3. Outlier-Robust Standardisation

For each time series we apply the weighted moving-average smoother
(`weighted_moving_stats`) row-by-row to produce:

- `prom_movil` â€” weighted moving average of neighbours
- `desv_movil` â€” weighted moving standard deviation
- `adjusted_value` â€” original value clipped to [mean âˆ’ Ïƒ, mean + Ïƒ]

We then model **both** the raw `CS` and the `adjusted_value` columns,
choosing the one that yields the lower combined metric score.

In [ ]:
def apply_standardization(group_df: pd.DataFrame) -> pd.DataFrame:
    """Apply weighted moving-average standardisation to one time series."""
    group_df = group_df.sort_values("Date").copy()
    sales = group_df["CS"].tolist()
    prom, desv, adjusted = [], [], []

    for i in range(len(sales)):
        p, d, a = weighted_moving_stats(i, sales, window_size=3)
        prom.append(p)
        desv.append(d)
        adjusted.append(a)

    group_df["prom_movil"] = prom
    group_df["desv_movil"] = desv
    group_df["adjusted_value"] = adjusted
    return group_df


df_std = (
    df_filled
    .groupby(["Country", "SKU"], group_keys=False)
    .apply(apply_standardization)
    .reset_index(drop=True)
)

print(f"Standardised DataFrame shape: {df_std.shape}")
df_std.head(8)

In [ ]:
# Visualise original vs adjusted for one series
sample = df_std[(df_std["SKU"] == 1001) & (df_std["Country"] == "US")].sort_values("Date")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sample["Date"], sample["CS"], label="Raw CS", color="steelblue", lw=1.5)
ax.plot(sample["Date"], sample["adjusted_value"], label="Adjusted", color="coral",
        lw=1.5, linestyle="--")
ax.fill_between(
    sample["Date"],
    sample["prom_movil"] - sample["desv_movil"],
    sample["prom_movil"] + sample["desv_movil"],
    alpha=0.15, color="grey", label="Â±1Ïƒ band"
)
ax.set_title("SKU 1001 â€” US: Raw vs Adjusted Demand", fontsize=12)
ax.set_ylabel("CS (units)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Bayesian Optimisation â€” Finding the Best ARIMA Order

We use **Optuna's TPE sampler** to search the `(p, d, q)` space for a
single representative series (SKU 1001, US).  The objective minimised is:

$$\text{combined} = 0.7 \times \text{sMAPE} + 0.3 \times \text{RMSLE}$$

Key settings:
- Search space: `p âˆˆ [0, 4]`, `d âˆˆ [0, 2]`, `q âˆˆ [0, 4]` (reduced for demo speed)
- Budget: **20 trials** (use 50 in production)
- Warm starts: `(1,1,1)` and `(1,0,0)` are evaluated first

> **Runtime note**: 20 trials on 36 months of data typically takes < 30 seconds.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Extract the series for SKU 1001, US
mask = (df_std["SKU"] == 1001) & (df_std["Country"] == "US")
series_df = df_std[mask].sort_values("Date")
series_values = series_df["CS"].values

print(f"Series length: {len(series_values)} months")
print(f"Series stats — mean: {series_values.mean():.1f}, std: {series_values.std():.1f}")
print()
print("Running Bayesian Optimisation (20 trials)...")

best_params, best_score = optimize_arima(
    series=series_values,
    p_range=(0, 3),
    d_range=(0, 2),
    q_range=(0, 3),
    P_range=(0, 2),
    D_range=(0, 1),
    Q_range=(0, 2),
    m=12,
    n_calls=20,
    n_jobs=1,
)

# BREAKING CHANGE: seasonal parameters now included in returned dict.
# All call sites must unpack P, D, Q, m in addition to p, d, q.
p_ = best_params["p"]
d_ = best_params["d"]
q_ = best_params["q"]
P_ = best_params["P"]
D_ = best_params["D"]
Q_ = best_params["Q"]
m_ = best_params["m"]

print(f"\nBest SARIMA order: ({p_},{d_},{q_})({P_},{D_},{Q_})[{m_}]")
print(f"Best combined metric score: {best_score:.4f}")


### Optimisation Convergence Curve

The convergence curve shows how the best score improves as more trials are
evaluated.  A rapid drop followed by a plateau indicates the optimiser has
found the near-optimal region efficiently.

In [ ]:
from optuna.samplers import TPESampler

from sarima_bayes.optimizer import OPTIMIZER_PENALTY, _evaluate_arima

# Re-run with a study object we can inspect for the convergence plot
sampler = TPESampler(seed=42, multivariate=True)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.enqueue_trial({"p": 1, "d": 1, "q": 1, "P": 1, "D": 1, "Q": 1})
study.enqueue_trial({"p": 1, "d": 0, "q": 0, "P": 0, "D": 0, "Q": 0})

def objective(trial):
    p = trial.suggest_int("p", 0, 3)
    d = trial.suggest_int("d", 0, 2)
    q = trial.suggest_int("q", 0, 3)
    P = trial.suggest_int("P", 0, 2)
    D = trial.suggest_int("D", 0, 1)
    Q = trial.suggest_int("Q", 0, 2)
    if (p + q) > 4 or (P + Q) > 3:
        return OPTIMIZER_PENALTY
    _, _, _, score, _ = _evaluate_arima(series_values, p, d, q, P, D, Q, m=12)
    return score

study.optimize(objective, n_trials=20, n_jobs=1)

# Extract trial scores
trial_scores = [t.value for t in study.trials if t.value is not None]
best_so_far  = np.minimum.accumulate(trial_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw scores per trial
axes[0].scatter(range(len(trial_scores)), trial_scores, s=40, alpha=0.6,
                color="steelblue", label="Trial score")
axes[0].plot(best_so_far, color="coral", lw=2, label="Best so far")
axes[0].set_xlabel("Trial #")
axes[0].set_ylabel("Combined metric")
axes[0].set_title("Optimisation Convergence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: parameter distribution of all trials
p_vals = [t.params["p"] for t in study.trials]
q_vals = [t.params["q"] for t in study.trials]
scores = [t.value for t in study.trials]

sc = axes[1].scatter(p_vals, q_vals, c=scores, cmap="RdYlGn_r", s=60, alpha=0.8)
plt.colorbar(sc, ax=axes[1], label="Score (lower=better)")
axes[1].set_xlabel("p (AR order)")
axes[1].set_ylabel("q (MA order)")
axes[1].set_title("(p, q) Search Space — coloured by score")
axes[1].grid(True, alpha=0.3)

best = study.best_params
label = f"Best: ({best['p']},{best['d']},{best['q']})({best['P']},{best['D']},{best['Q']})[12]"
axes[1].scatter([best["p"]], [best["q"]], marker="*", s=300, color="black",
                zorder=5, label=label)
axes[1].legend(fontsize=8)

plt.suptitle("Bayesian Optimisation — SKU 1001, US", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. SARIMA Forecast

Using the best `(p, d, q)` found above, we fit the ARIMA model on the full
historical series and generate a **12-month ahead** forecast with 95 %
confidence intervals.

In [ ]:
N_FORECAST = 12
p_  = best_params["p"]
d_  = best_params["d"]
q_  = best_params["q"]
P_  = best_params["P"]
D_  = best_params["D"]
Q_  = best_params["Q"]
m_  = best_params["m"]

# BREAKING CHANGE: seasonal parameters now included in returned dict.
# All call sites must unpack P, D, Q, m in addition to p, d, q.
forecast_df, conf_int, _, _, forecast_series = pred_arima(
    bd=series_df[["Date", "CS"]],
    col_x="Date",
    col_y="CS",
    order=(p_, d_, q_),
    s_order=(P_, D_, Q_, m_),
    n_per=N_FORECAST,
    alpha=0.05,
)

# Clip negatives — demand cannot be negative
forecast_df["Predictions"] = forecast_df["Predictions"].clip(lower=0)

print(f"SARIMA({p_},{d_},{q_})({P_},{D_},{Q_})[{m_}] — {N_FORECAST}-month forecast:")
print(forecast_df[["Predictions", "Std Error"]].round(2))


In [ ]:
# â”€â”€ Forecast Plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(13, 5))

# Historical data
ax.plot(series_df["Date"], series_df["CS"],
        label="Historical", color="steelblue", lw=2, marker="o", ms=3)

# Forecast point estimate
ax.plot(forecast_df.index, forecast_df["Predictions"],
        label=f"Forecast SARIMA({p_},{d_},{q_})({P_},{D_},{Q_})[{m_}]", color="coral", lw=2, marker="s", ms=4)

# Confidence interval shading
ci_lower = conf_int.iloc[:, 0].clip(lower=0)
ci_upper = conf_int.iloc[:, 1]
ax.fill_between(forecast_df.index, ci_lower, ci_upper,
                alpha=0.2, color="coral", label="95% CI")

# Vertical separator between history and forecast
ax.axvline(series_df["Date"].max(), color="grey", linestyle="--", lw=1, alpha=0.7)
ax.text(series_df["Date"].max(), ax.get_ylim()[1] * 0.95,
        " Forecast â†’", fontsize=9, color="grey")

ax.set_title(
    f"SKU 1001 — US: 12-Month Forecast "
    f"SARIMA({p_},{d_},{q_})({P_},{D_},{Q_})[{m_}]",
    fontsize=13, fontweight="bold")
ax.set_ylabel("CS (units)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Batch Pipeline â€” All SKU / Country Combinations

In production, the pipeline runs for every `(Country, SKU)` combination.
Here we demonstrate a compact loop over all six series.

> **Performance tip**: wrap the loop with
> `joblib.Parallel(n_jobs=-1, backend="threading")`
> to process all series in parallel.

In [ ]:
results = []  # will collect one row per (Country, SKU)

for (country, sku), grp in df_std.groupby(["Country", "SKU"]):
    grp = grp.sort_values("Date")
    series = grp["CS"].values

    # Find best SARIMA params (reduced budget for demo)
    # BREAKING CHANGE: seasonal parameters now included in returned dict.
    # All call sites must unpack P, D, Q, m in addition to p, d, q.
    params, score = optimize_arima(series, p_range=(0, 3), d_range=(0, 2),
                                   q_range=(0, 3), P_range=(0, 2),
                                   D_range=(0, 1), Q_range=(0, 2),
                                   m=12, n_calls=15, n_jobs=1)
    p, d, q = params["p"], params["d"], params["q"]
    P, D, Q, m_val = params["P"], params["D"], params["Q"], params["m"]

    # Generate 12-month forecast (passing seasonal order)
    fcst = forecast_arima(grp[["Date", "CS"]], "Date", "CS",
                          p, d, q, 12, country, sku,
                          P=P, D=D, Q=Q, m=m_val)

    results.append({
        "Country": country,
        "SKU": sku,
        "p": p, "d": d, "q": q,
        "P": P, "D": D, "Q": Q, "m": m_val,
        "score": round(score, 4),
        "forecast": fcst,
    })
    print(f"  {country} | SKU {sku} → SARIMA({p},{d},{q})({P},{D},{Q})[{m_val}] | score={score:.4f}")

print("\nDone.")


In [ ]:
# Summary table
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k != "forecast"}
    for r in results
])
summary

### Forecast comparison â€” all series

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=False)
axes = axes.flatten()

for ax, result in zip(axes, results):
    country = result["Country"]
    sku = result["SKU"]
    p, d, q = result["p"], result["d"], result["q"]
    fcst = result["forecast"]

    hist = df_std[(df_std["SKU"] == sku) & (df_std["Country"] == country)].sort_values("Date")

    ax.plot(hist["Date"], hist["CS"], color="steelblue", lw=1.5, label="History")
    ax.plot(fcst["Date"], fcst["Pred"], color="coral", lw=1.5, linestyle="--",
            marker="s", ms=3, label=f"ARIMA({p},{d},{q})")
    ax.axvline(hist["Date"].max(), color="grey", linestyle=":", lw=0.8)
    ax.set_title(f"SKU {sku} â€” {country} | score={result['score']}", fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)

fig.suptitle("12-Month Forecasts â€” All Series", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Export Results

Consolidate all forecasts into a single DataFrame and optionally export to
Excel (requires `openpyxl`).

In [ ]:
all_forecasts = pd.concat([r["forecast"] for r in results], ignore_index=True)
print(f"Total forecast rows: {len(all_forecasts)}")
all_forecasts.head(12)

In [ ]:
# Uncomment to export to Excel:
# output_dir = "../data/output"
# os.makedirs(output_dir, exist_ok=True)
# all_forecasts.to_excel(f"{output_dir}/demo_forecast.xlsx", index=False)
# summary.to_excel(f"{output_dir}/demo_metrics.xlsx", index=False)
# print("Exported.")

print("Pipeline complete!")
print(f"Forecasted {len(results)} series Ã— 12 months = {len(all_forecasts)} forecast rows.")

## Walk-Forward Validation

In [ ]:
from sarima_bayes import optimize_arima
from sarima_bayes.model import pred_arima
from sarima_bayes.validation import walk_forward_validation


def make_sarima_fn(n_calls=20, n_per=12):
    """Factory: returns a SARIMA+BO model_fn for use in walk_forward_validation."""
    def sarima_fn(train: pd.Series) -> pd.Series:
        # BREAKING CHANGE: seasonal parameters now included in returned dict.
        # All call sites must unpack P, D, Q, m in addition to p, d, q.
        best_params, _ = optimize_arima(train.values, n_calls=n_calls)
        bd = train.reset_index()
        bd.columns = ["Date", "CS"]
        *_, forecast_series = pred_arima(
            bd, "Date", "CS",
            order=(best_params["p"], best_params["d"], best_params["q"]),
            s_order=(best_params["P"], best_params["D"], best_params["Q"], best_params["m"]),
            n_per=n_per,
        )
        return forecast_series
    return sarima_fn


# Run walk-forward validation on one SKU/Country series
mask_us = (df_std["SKU"] == 1001) & (df_std["Country"] == "US")
grp_us = df_std[mask_us].sort_values("Date")
wfv_series = pd.Series(
    grp_us["CS"].values,
    index=pd.DatetimeIndex(grp_us["Date"]),
)
wfv_series.index.freq = "MS"

wfv_result = walk_forward_validation(
    wfv_series,
    make_sarima_fn(n_calls=10, n_per=6),
    n_folds=3,
    test_size=6,
    min_train_size=12,
)
print(wfv_result)


In [ ]:
# Bar chart: sMAPE per fold
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(wfv_result["fold"].astype(str), wfv_result["sMAPE"], color="steelblue", width=0.5)
ax.set_xlabel("Fold")
ax.set_ylabel("sMAPE (%)")
ax.set_title("sMAPE per Fold — SARIMA+BO")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Benchmark Comparison

In [ ]:
from sarima_bayes.benchmarks import run_benchmark_comparison

# Build a small synthetic DataFrame with enough history for 3 folds
# (min_train_size=12 + 3*3 = 21 months needed per group; we use df_std which has 36)
benchmark_results = run_benchmark_comparison(
    df_std,
    group_cols=["Country", "SKU"],
    target_col="CS",
    date_col="Date",
    sarima_model_fn=make_sarima_fn(n_calls=10, n_per=3),
    n_folds=3,
    test_size=3,
    min_train_size=12,
)
print(benchmark_results.head(12))


In [ ]:
from sarima_bayes.benchmarks import summary_table

tbl = summary_table(benchmark_results, group_cols=["Country", "SKU"])
tbl.style.highlight_min(subset=["sMAPE_mean"], axis=0, color="lightgreen")


## Interpreting the Benchmark Table

A few things to keep in mind when reading the results above:

- **Short series, low signal**: With only 36 months of history and a 3-fold CV,
  the training windows are small (~12–21 data points). Statistical models are
  less stable at this scale, so it is entirely expected that SARIMA+BO may not
  beat the seasonal naïve baseline on every series.

- **Not a failure**: A  result does not mean Bayesian
  Optimisation adds no value. It means that for *this series*, at *this history
  length*, the complexity of ARIMA is not justified. That is useful information.

- **Primary use of the table**: Identify which SKU/Country combinations benefit
  most from BO-tuned models (large gap vs. naïve), and allocate your compute
  budget accordingly. Use simpler baselines for low-signal series.

- **Production recommendation**: Run the comparison on at least 36–60 months of
  real data per series with  for more reliable estimates.
